# Servidor de Inferencia de Respaldo en Google Colab

Este notebook levanta exactamente el mismo endpoint único (`/inferencia`) en Google Colab y lo expone por HTTPS público mediante Cloudflare Quick Tunnel (`trycloudflare.com`).

Para alternar entre RunPod y Colab, únicamente se reemplaza la variable `INFERENCE_URL` en el backend web.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Clonar repositorio del proyecto (reemplazar URL por su repo remoto)
!git clone https://github.com/TU_USUARIO/proyecto_caseta_IA.git /content/proyecto_caseta_IA || true
%cd /content/proyecto_caseta_IA/inferencia
!pip install -q -r requirements.txt
!python diagnostico_entorno.py

In [ ]:
# Instalar cloudflared para obtener URL HTTPS pública sin depender de recursos locales
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import os, subprocess, time, re

os.environ['MODEL_M1_PLATES_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m1_placas.pt'
os.environ['MODEL_M2_VEHICLE_SEG_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m2_vehiculos_seg.pt'
os.environ['MODEL_M3_WHEEL_POSE_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m3_ruedas_pose.pt'
os.environ['MODEL_M4_PASSENGERS_SEG_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m4_pasajeros_seg.pt'
os.environ['INFERENCE_API_KEY'] = 'CAMBIAR_MISMO_TOKEN_QUE_EN_WEB'

server = subprocess.Popen(['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(6)

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
for line in tunnel.stdout:
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n====================================================')
        print('COPIAR ESTA URL EN INFERENCE_URL DEL BACKEND WEB:')
        print(m.group(0))
        print('====================================================\n')
        break
server.wait()